In [1]:
from connection.spark_session import get_spark
from config.config import CATALOG_NAME

spark = get_spark("VerifyIncrementalLoad")

In [2]:
spark.read.table(f"{CATALOG_NAME}.warehouse.fact_orders") \
    .filter("updated_at > '2024-12-31'") \
    .select("order_id", "status", "updated_at") \
    .show(20, truncate=False)

+------------+---------+-------------------+
|order_id    |status   |updated_at         |
+------------+---------+-------------------+
|ORD-14165DB8|pending  |2025-01-01 00:00:00|
|ORD-1DB4C892|delivered|2025-01-01 00:00:00|
|ORD-34786BC9|confirmed|2025-01-01 00:00:00|
|ORD-5C3F8D6A|pending  |2025-01-01 00:00:00|
|ORD-8BDDEAE2|shipped  |2025-01-01 00:00:00|
|ORD-9BACF389|confirmed|2025-01-01 00:00:00|
|ORD-9CC9C6FA|shipped  |2025-01-01 00:00:00|
|ORD-B3684CE0|pending  |2025-01-01 00:00:00|
|ORD-BC9B58FE|pending  |2025-01-01 00:00:00|
|ORD-CC227C0B|confirmed|2025-01-01 00:00:00|
|ORD-CD22748D|confirmed|2025-01-01 00:00:00|
|ORD-D78D096C|confirmed|2025-01-01 00:00:00|
|ORD-F2A3E6D2|confirmed|2025-01-01 00:00:00|
|ORD-F5F07173|pending  |2025-01-01 00:00:00|
|ORD-F821BD95|confirmed|2025-01-01 00:00:00|
+------------+---------+-------------------+



In [3]:
spark.sql(f"SELECT partition, record_count FROM {CATALOG_NAME}.warehouse.fact_orders.partitions ORDER BY partition").show(50, truncate=False)

+---------+------------+
|partition|record_count|
+---------+------------+
|{635}    |3           |
|{636}    |44          |
|{637}    |31          |
|{638}    |43          |
|{639}    |43          |
|{640}    |51          |
|{641}    |48          |
|{642}    |53          |
|{643}    |53          |
|{644}    |58          |
|{645}    |37          |
|{646}    |55          |
|{647}    |62          |
|{648}    |60          |
|{649}    |51          |
|{650}    |39          |
|{651}    |63          |
|{652}    |39          |
|{653}    |67          |
|{654}    |64          |
|{655}    |50          |
|{656}    |45          |
|{657}    |39          |
|{658}    |50          |
|{659}    |57          |
+---------+------------+



In [2]:
spark.read.table(f"{CATALOG_NAME}.warehouse.fact_orders") \
    .selectExpr("date_format(created_at, 'yyyy-MM') as month") \
    .groupBy("month").count().orderBy("month").show(50)

+-------+-----+
|  month|count|
+-------+-----+
|2023-01|   46|
|2023-02|   32|
|2023-03|   41|
|2023-04|   42|
|2023-05|   53|
|2023-06|   48|
|2023-07|   49|
|2023-08|   54|
|2023-09|   61|
|2023-10|   38|
|2023-11|   52|
|2023-12|   59|
|2024-01|   66|
|2024-02|   48|
|2024-03|   42|
|2024-04|   60|
|2024-05|   41|
|2024-06|   65|
|2024-07|   66|
|2024-08|   48|
|2024-09|   47|
|2024-10|   40|
|2024-11|   49|
|2024-12|   53|
|2025-01|    8|
+-------+-----+



In [3]:
spark.sql(f"SELECT committed_at, operation, summary['added-data-files'] as files_added FROM local.warehouse.fact_orders.snapshots ORDER BY committed_at").show(60, truncate=False)

+-----------------------+---------+-----------+
|committed_at           |operation|files_added|
+-----------------------+---------+-----------+
|2026-07-01 06:35:43.634|overwrite|25         |
|2026-07-01 06:36:37.348|overwrite|1          |
|2026-07-01 06:36:41.55 |overwrite|2          |
|2026-07-01 06:36:45.485|overwrite|2          |
|2026-07-01 06:36:49.252|overwrite|2          |
|2026-07-01 06:36:52.602|overwrite|2          |
|2026-07-01 06:36:55.849|overwrite|2          |
|2026-07-01 06:36:59.538|overwrite|2          |
|2026-07-01 06:37:02.796|overwrite|2          |
|2026-07-01 06:37:06.415|overwrite|2          |
|2026-07-01 06:37:09.448|overwrite|2          |
|2026-07-01 06:37:11.748|overwrite|2          |
|2026-07-01 06:37:14.925|overwrite|2          |
|2026-07-01 06:37:17.318|overwrite|2          |
|2026-07-01 06:37:20.401|overwrite|2          |
|2026-07-01 06:37:22.754|overwrite|2          |
|2026-07-01 06:37:24.969|overwrite|1          |
|2026-07-01 06:37:28.198|overwrite|1    

In [7]:
spark.sql("""SELECT COUNT(*) FROM local.warehouse.fact_orders.manifests""").show()

+--------+
|count(1)|
+--------+
|       6|
+--------+



In [3]:
import os
from pathlib import Path

# Adjust this if your warehouse path differs
warehouse_root = Path("D:/tmp/warehouse/warehouse/fact_orders/data")

print(f"Scanning: {warehouse_root}\n")

if not warehouse_root.exists():
    print("Path does not exist -- check your actual warehouse location (e.g. under 'warehouse' namespace folder).")
else:
    parquet_files = sorted(warehouse_root.rglob("*.parquet"))
    total_size = 0
    for f in parquet_files:
        size_kb = f.stat().st_size / 1024
        total_size += size_kb
        print(f"{size_kb:8.1f} KB  {f.relative_to(warehouse_root)}")
    print(f"\nTotal physical parquet files on disk: {len(parquet_files)}")
    print(f"Total size: {total_size:.1f} KB")

Scanning: D:\tmp\warehouse\warehouse\fact_orders\data

    29.6 KB  00000-10-b6ec82b5-4357-47ce-bde1-d45cc029496d-0-00001.parquet
    29.8 KB  00000-106-87cb3e24-119b-41a1-9d1b-6bc97db4c50f-0-00001.parquet
    29.9 KB  00000-126-79bc0b1c-f028-44f7-92ee-51d863c9e023-0-00001.parquet
    30.0 KB  00000-142-d4a6352c-f69d-42ed-a023-b5bb09ff6f5c-0-00001.parquet
    30.1 KB  00000-159-3ceb2c94-b644-4ff7-8ba5-45e9cf21fceb-0-00001.parquet
    30.1 KB  00000-180-c6fc4012-5f4b-48bb-a18c-4660e453c7d7-0-00001.parquet
    30.1 KB  00000-197-cb1184ea-ccdc-4670-be1a-8fe21ea5451c-0-00001.parquet
    30.2 KB  00000-214-f9c09f9e-0d4a-43b4-b9d5-65f4aca72fff-0-00001.parquet
    30.3 KB  00000-232-71024573-0959-43dc-a624-9d2637df0f79-0-00001.parquet
    29.6 KB  00000-25-472dd79d-228a-49de-98ef-e36a95ca03c9-0-00001.parquet
    30.4 KB  00000-250-ba3099b6-488b-4247-ac80-6191a82317ee-0-00001.parquet
    30.4 KB  00000-268-91ab8c76-52d3-49c5-82b9-e30f5711c917-0-00001.parquet
    30.5 KB  00000-286-e252aad2-228

In [4]:
spark.sql("SELECT file_path, file_size_in_bytes FROM local.warehouse.fact_orders.files").show(truncate=False)

+--------------------------------------------------------------------------------------------------------+------------------+
|file_path                                                                                               |file_size_in_bytes|
+--------------------------------------------------------------------------------------------------------+------------------+
|/tmp/warehouse/warehouse/fact_orders/data/00000-806-78810081-f0c9-4350-9455-1b75905b2c9e-0-00001.parquet|32828             |
+--------------------------------------------------------------------------------------------------------+------------------+



In [5]:
spark.sql("SELECT snapshot_id, committed_at, operation, summary FROM local.warehouse.fact_orders.snapshots ORDER BY committed_at DESC LIMIT 3").show(truncate=False)

+-------------------+-----------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
spark.sql("""
SELECT file_path,
       file_size_in_bytes
FROM local.warehouse.fact_orders.files
""").show(100, truncate=False)

+---------------------------------------------------------------------------------------------------------------+------------------+
|file_path                                                                                                      |file_size_in_bytes|
+---------------------------------------------------------------------------------------------------------------+------------------+
|/tmp/warehouse/warehouse/fact_orders/data/00000-299-24688ba6-ff35-4d5b-af89-ac998da83df8-0-00001.parquet       |32877             |
|/tmp/warehouse/warehouse/fact_orders/data/00000-1052-6f910fcb-4995-4d01-9019-6f6d42c08b1a-00001-deletes.parquet|1513              |
|/tmp/warehouse/warehouse/fact_orders/data/00000-1052-6f910fcb-4995-4d01-9019-6f6d42c08b1a-00002-deletes.parquet|1499              |
|/tmp/warehouse/warehouse/fact_orders/data/00000-1052-6f910fcb-4995-4d01-9019-6f6d42c08b1a-00003-deletes.parquet|1451              |
|/tmp/warehouse/warehouse/fact_orders/data/00000-1052-6f910fcb-4995-4

In [8]:
spark.sql("""
SHOW TBLPROPERTIES local.warehouse.fact_orders
""").show(200, truncate=False)

+-------------------------------+-------------------+
|key                            |value              |
+-------------------------------+-------------------+
|current-snapshot-id            |3191331259433771885|
|format                         |iceberg/parquet    |
|format-version                 |2                  |
|write.delete.mode              |merge-on-read      |
|write.merge.mode               |merge-on-read      |
|write.parquet.compression-codec|zstd               |
|write.update.mode              |merge-on-read      |
+-------------------------------+-------------------+



In [3]:
spark.table(f"{CATALOG_NAME}.warehouse.dim_customer").printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- tier: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- customer_sk: long (nullable = true)

